この教材では、コンピュータがすべてを **0 と 1（ビット）** で扱っていること、そしてその「バイナリ」の知識が **セキュリティ** にどうつながるかを学びます。

前半で **2進数・16進数・ビット演算・文字コード・バイト列** という土台を作り、後半でそれを武器に **XOR暗号・エンコード・ハッシュ** を体験します。

最後の総仕上げでは、それらを組み合わせて、**「届いたパケットのマスクを解き、改ざんを検知し、中身を読み取る」** ミニ復号ツールを自分で完成させます。

`____`（アンダースコア）や `pass` の部分は、自分で書き換える場所です。教材パートの実行例はまず読んで、動かして、仕組みを確かめてから練習問題に進みましょう。

# 前半: バイナリの土台

## 2進数と16進数

人は普段10進数（0〜9）を使いますが、コンピュータは内部を **2進数**（0 と 1 だけ）で表しています。ただ2進数は桁が長くて読みにくいので、**16進数**（0〜9, a〜f）もよく使われます。

Python では数値の前に記号を付けて、これらを直接書けます。

- `0b` … 2進数（**b**inary）。各桁は 2 の n 乗を表す。`0b1111` = 8+4+2+1 = 15
- `0x` … 16進数（he**x**adecimal）。各桁は 16 の n 乗を表す。**1桁がちょうど4ビット** に対応するので、1バイト（8ビット）を2桁で表せて便利

In [ ]:
# 0b（2進数）と 0x（16進数）で数を書いてみる
print(0b100)              # 4
print(0b1111, "=", 8 + 4 + 2 + 1)   # 15 = 15
print(0x41)               # 65
print(0xff)               # 255  ← 1バイトで表せる最大値

# 0b / 0x で書いても、中身はただの整数。10進数として表示される
print(0b1010 == 10)       # True
print(0x0a == 10)         # True

## 基数を変換する（`bin` / `hex` / `int`）

同じ数でも、見せ方（基数）はいくつもあります。`0b1010` のように **手で書く** のではなく、**変数の値を** 2進や16進の表示に変えたいときは、変換関数を使います。

- `bin(x)` … 整数を「2進数の**文字列**」に（例 `'0b1010'`）
- `hex(x)` … 整数を「16進数の**文字列**」に（例 `'0xa'`）
- `int("1010", 2)` / `int("ff", 16)` … 文字列を整数に戻す（第2引数が基数）

`bin` / `hex` が返すのは **文字列** である点がポイントです。

In [ ]:
print(bin(10))          # 0b1010   ← 文字列
print(hex(255))         # 0xff     ← 文字列
print(int("1010", 2))   # 10       （2進の文字列 → 整数）
print(int("ff", 16))    # 255      （16進の文字列 → 整数）

## ビット演算子

2進数で表した数の **各ビット** に対して計算するのがビット演算子です。

| 演算子 | 名前 | はたらき |
|:---:|---|---|
| `&` | AND（論理積） | 両方が1のとき1 |
| `\|` | OR（論理和） | どちらかが1なら1 |
| `^` | XOR（排他的論理和） | **違えば1、同じなら0** |
| `~` | NOT（反転） | 0と1を反転 |
| `<<` | 左シフト | ビットを左へ（×2ずつ） |
| `>>` | 右シフト | ビットを右へ（÷2ずつ） |

特に **XOR（`^`）** は後半の主役です。「同じ値で2回XORすると元に戻る」という性質を覚えておきましょう。

In [ ]:
print(bin(0b1100 | 0b1010))   # OR  -> 0b1110
print(bin(0b1100 & 0b1010))   # AND -> 0b1000
print(bin(0b1100 ^ 0b1010))   # XOR -> 0b110
print(1 << 4)                 # 左シフト -> 16
print(240 >> 4)               # 右シフト -> 15

# XORの大事な性質: 同じ値で2回XORすると元に戻る
print(5 ^ 3 ^ 3)              # 5

## 文字コード（文字 ⇔ 数）

コンピュータは数しか扱えないので、文字にも数を割り当てています。これが **文字コード** です（英数字記号の **ASCII**、世界中の文字を含む **Unicode**）。

- `ord(文字)` … 文字 → 数
- `chr(数)` … 数 → 文字

In [ ]:
print(ord('A'))                 # 65
print(hex(ord('A')))            # 0x41
print(chr(65))                  # A
print([ord(c) for c in "Hi"])   # [72, 105]

## バイト列（`bytes`）

ビットを8個まとめた **バイト**（0〜255）が、実データの基本単位です。ファイルの中身も、ネットワークを流れるパケットも、正体は **バイト列** です。Python では `bytes` 型で扱います。

- `bytes([192, 168, 0, 1])` … 数のリストからバイト列を作る
- `.hex()` … バイト列 → 16進文字列 / `bytes.fromhex(...)` … その逆
- `"文字".encode("utf-8")` / `バイト列.decode("utf-8")` … 文字列 ⇔ バイト列

In [ ]:
data = bytes([192, 168, 0, 1])   # IPアドレスをバイト列で
print(data[0])                   # 192
print(data.hex())                # c0a80001
print(bytes.fromhex("c0a80001")) # b'\xc0\xa8\x00\x01'

# 文字列とバイト列の変換（日本語は UTF-8 で複数バイトになる）
print("あ".encode("utf-8").hex())            # e38182
print(bytes.fromhex("e38182").decode("utf-8"))  # あ

# 後半: セキュリティへの応用

前半の土台（ビット演算・文字コード・バイト列）を使って、セキュリティの基本を体験します。

## XOR暗号 — ビット演算 `^` の応用

XOR には **`a ^ b ^ b == a`**（同じ値で2回XORすると元に戻る）という性質があります。
つまり、鍵でXORして暗号化し、**同じ鍵でもう一度XORすれば復号** できます。暗号化と復号が **まったく同じ処理** になるのがXOR暗号です。

> この XOR は遊びではありません。次回学ぶ **WebSocket** の通信では、ブラウザが送るデータが XOR で **マスク** されています。マスク鍵で XOR を解けば中身が読めます。

In [ ]:
def xor_cipher(data, key):
    """data(bytes) の各バイトを key とXORする。暗号化・復号 兼用。"""
    return bytes(b ^ key for b in data)

secret = xor_cipher(b"HELLO", 0x2a)
print(secret.hex())                       # 暗号化された読めないバイト列
print(xor_cipher(secret, 0x2a).decode())  # HELLO  ← 同じ鍵・同じ関数で戻る

## エンコード（base64）

画像や暗号文のような **バイナリ** を、メールやURLのような「文字しか通らない場所」で運びたいことがあります。バイナリを **文字だけ** で表し直すのが **base64** です。

※ base64 は **暗号ではありません**。鍵なしで誰でも元に戻せます。あくまで「運ぶための変換」です。

In [ ]:
import base64

enc = base64.b64encode("秘密のメモ".encode("utf-8"))
print(enc.decode())                          # base64の文字列
print(base64.b64decode(enc).decode("utf-8")) # 秘密のメモ

## ハッシュ（SHA-256）

**ハッシュ関数** は、どんなデータからも決まった長さの「指紋」を作ります。

- **一方向**: データ→ハッシュは簡単。ハッシュ→元データは実質不可能
- **同じ入力なら必ず同じ出力**
- **アバランシェ効果**: 入力が **1文字** 違うだけで、出力は **まるごと** 変わる

用途は **パスワードの保存**（元を保存せず指紋だけ持つ）や **改ざん検知**（指紋を比べる）です。

In [ ]:
import hashlib

print(hashlib.sha256(b"password").hexdigest())

# アバランシェ効果: 末尾1文字だけ違う2つを比べる
h1 = hashlib.sha256(b"password").digest()
h2 = hashlib.sha256(b"passworE").digest()
diff = sum(bin(a ^ b).count("1") for a, b in zip(h1, h2))
print("1文字違いで 256ビット中", diff, "ビットが変化")

## バイト列でパケットを組む・解く（`struct`）

通信では、複数の値を **決まった並びのバイト列**（パケット）にまとめて送ります。`struct` で、数値とバイト列を相互変換できます。

書式 `">BHhh"` の意味: `>`=ビッグエンディアン、`B`=1バイト符号なし、`H`=2バイト符号なし、`h`=2バイト符号あり。
例として「バージョン・ID・x座標・y座標」を1つのパケットにします（対戦ゲームの通信そのものです）。

In [ ]:
import struct

packet = struct.pack(">BHhh", 1, 7, 100, -20)  # version=1, id=7, x=100, y=-20
print(packet.hex())                            # バイト列（7バイト）

version, pid, x, y = struct.unpack(">BHhh", packet)
print(version, pid, x, y)                      # 1 7 100 -20

# 練習問題

ここからは自分で書いてみましょう。上の教材パートを見返しながらで大丈夫です。

## 練習問題1: 10・50・100 を3つの基数で表す

`for` ループで、各数を **10進・2進・16進** の表示にして出力してください。

**ヒント**: 変数の値を2進・16進の**文字列**にするには `bin()` / `hex()` を使います（`0b`/`0x` のリテラルは変数には使えません）。

**期待される出力**:
```
10 -> 10進:10  2進:0b1010  16進:0xa
50 -> 10進:50  2進:0b110010  16進:0x32
100 -> 10進:100  2進:0b1100100  16進:0x64
```

In [ ]:
for n in [10, 50, 100]:
    decimal = ____   # n の10進数（そのまま）
    binary  = ____   # n を2進数の文字列に
    hexad   = ____   # n を16進数の文字列に
    print("{} -> 10進:{}  2進:{}  16進:{}".format(n, decimal, binary, hexad))

## 練習問題2: ビット演算とマスク

(1) `a = 12`, `b = 25` の **OR** と **AND** を求めてください。
(2) IPアドレスの第4オクテット `245`（`0b11110101`）から、**上位4ビット** と **下位4ビット** を取り出してください。

**ヒント**: 取り出したいビットだけを1にした **マスク** を作り、`&`（AND）を取ります。上位4ビットのマスクは `0b11110000`、下位4ビットは `0b00001111`。

**期待される出力**:
```
OR: 29
AND: 8
上位: 0b11110000
下位: 0b101
```

In [ ]:
a = 12
b = 25
print("OR:", ____)
print("AND:", ____)

octet4 = 0b11110101
high4 = ____   # 上位4ビットを取り出す
low4  = ____   # 下位4ビットを取り出す
print("上位:", bin(high4))
print("下位:", bin(low4))

## 練習問題3: 文字コードのリストを文字列に戻す

ユニコードの数のリストを受け取り、文字列にして返す関数 `convert_to_str` を完成させてください。

**ヒント**: 各数を `chr()` で文字にして、つなげます。

**期待される動作**:
```python
convert_to_str([109, 105, 116, 115, 117, 121, 97])   # 'mitsuya'
```

In [ ]:
def convert_to_str(charcodes):
    # ここに書いてください
    pass

print(convert_to_str([109, 105, 116, 115, 117, 121, 97]))   # mitsuya

## 練習問題4: XOR暗号を総当りで破る

暗号文 `secret` は、ある **1バイトの鍵**（0〜255のどれか）でXORされています。鍵は分かりません。
すべての鍵を試して（総当り）、復号結果が `FLAG{` で始まる正しい鍵を見つけ、フラグを表示してください。

**ヒント**: 教材の `xor_cipher` を使います。`for key in range(256):` で全部試し、`.startswith(b"FLAG")` で判定します。

**期待される出力**: `鍵 0x5a -> FLAG{binary_hero}`

In [ ]:
def xor_cipher(data, key):
    return bytes(b ^ key for b in data)

secret = bytes.fromhex("1c161b1d213833343b282305323f283527")

for key in range(256):
    plain = xor_cipher(secret, key)
    if ____:                       # plain が b"FLAG" で始まるか
        print("鍵", hex(key), "->", plain.decode())
        break

## 練習問題5: ハッシュで改ざんを検知する

送信側は、メッセージと一緒にその **SHA-256ハッシュ**（指紋）を送ります。受信側は、受け取ったメッセージのハッシュを計算し直し、届いた指紋と一致するかで **改ざんの有無** を判定します。

`received`（受信メッセージ）のハッシュを計算し、`expected`（届いた指紋）と比べてください。

**ヒント**: `hashlib.sha256(文字列.encode()).hexdigest()` で指紋を作り、`==` で比較します。

**動作の確認**: そのまま実行すると「改ざんなし」。`received` を書き換えて実行すると「改ざんあり！」になります。

In [ ]:
import hashlib

message = "振込先: 口座A"
expected = hashlib.sha256(message.encode()).hexdigest()   # 送信側が付けた指紋

# --- 受信側 ---
received = "振込先: 口座A"      # ← ここを書き換えると改ざんを再現できる
actual = ____                   # received の SHA-256（hexdigest）
if ____:                        # actual と expected が一致するか
    print("改ざんなし")
else:
    print("改ざんあり！")

## 練習問題6（総仕上げ）: パケット復号ツールを作る

いよいよ総仕上げです。ゲームサーバから、**XORでマスクされたパケット** と、その **完全性ハッシュ** が届きました。前半・後半で学んだことを全部使って、中身を読み取ります。

届いたもの:
- `masked` … XORマスク済みのパケット（16進文字列）
- `key` … マスク鍵 `0x3c`
- `expected` … 元パケットのSHA-256（改ざん検知用）

次の3ステップを行う関数 `decode_packet` を完成させてください。

1. **マスクを解く**: `masked` を鍵 `key` でXORして元のバイト列に戻す
2. **改ざん検知**: 戻したバイト列のSHA-256が `expected` と一致するか確認（違えば「改ざんあり」と表示して終了）
3. **中身を読む**: `struct.unpack(">BHhh", ...)` で version・id・x・y を取り出して表示

**期待される出力**:
```
改ざんなし
ID=7 x=100 y=-20
```

In [ ]:
import hashlib, struct

def xor_cipher(data, key):
    return bytes(b ^ key for b in data)

masked = "3d3c3b3c58c3d0"
key = 0x3c
expected = "734e0eb454532c0178e16046e5e3354d6a1a9230721e689a9296c8ee7e2a9fbe"

def decode_packet(masked, key, expected):
    raw = ____                       # 1) masked(16進文字列)をバイト列にして、keyでXORして解く
    if ____:                         # 2) raw のSHA-256が expected と一致するか
        print("改ざんなし")
    else:
        print("改ざんあり！")
        return
    version, pid, x, y = ____        # 3) raw を ">BHhh" で unpack
    print("ID={} x={} y={}".format(pid, x, y))

decode_packet(masked, key, expected)

# おつかれさまでした

バイナリが読めると、セキュリティの世界が見えてきます。

- **覗く（sniff）**: 通信やファイルの中身は結局バイト列。`bytes` / `.hex()` で読める
- **隠す**: XOR暗号や base64 で中身を分からなくする（`^` / `base64`）
- **見抜く（detect）**: ハッシュで改ざん・すり替えを検知する（`hashlib`）
- **守る**: これらを理解して初めて、安全な仕組みを作れる

次回の **ネットワークプログラミング回** では、これらのバイト列が実際にネットワークを流れる様子（HTTP・WebSocket）を扱います。そして最後の **対戦ゲーム回** で、今日作った「パケットを読み解く力」を使って全員で競います。